### Tenacity - Retrying LLM calls

In [2]:
import random

def flaky_llm_call(prompt: str):
    if random.random() < 0.5: 
        raise Exception("429: Too many api calls")
    return f"LLM response to: {prompt}"

In [ ]:
# Without tenacity
def call_llm(prompt: str):
    return flaky_llm_call(prompt)

random.seed(1) # on 2 llm call is happening. not on 1 and 4

try:
    print(call_llm("What is gpt"))
except Exception as e:
    print(f"call failed: {str(e)}")

call failed: 429: Too many api calls


In [ ]:
# With tenacity
from tenacity import retry, stop_after_attempt, wait_fixed

@retry(
    stop= stop_after_attempt(3), # 3 retry calls only
    wait= wait_fixed(2),         # wait for 2 sec after every retry
    reraise= True                # overwrites tenacity's retry error with actual function error 
)
def call_llm_basic_retry(prompt: str):
    print("Attempting call") # one time it is always printed, second time print means error occured
    return flaky_llm_call(prompt)

random.seed(2)

call_llm_basic_retry("Provide me bun")

ModuleNotFoundError: No module named 'tenacity'

In [ ]:
# Exceptional backoff with tenacity: wait time increases exponentially
from tenacity import wait_exponential, RetryCallState

def log_retry(retry_state: RetryCallState) -> None:
    wait_time = retry_state.next_action.sleep # tenacity tells us how long it's about to sleep
    print(f"Attempt: {retry_state.attempt_number} failed -- retrying in {wait_time:.1f}s..")

attempts_so_far = {'count':0} # forces first 3 calls to fail, so we can watch backoff escalates

def flaky_llm_call_backoff(prompt str) -> None:
    attempts_so_far['count'] += 1
    if attempts_so_far['count'] < 4:
        raise Exception("429: Too many requests")
    return f"LLM response for: {prompt}"

@retry(
    stop= stop_after_attempt(5), # stop after 5 attempts
    # min wait=2sec, max wait=10sec, multiper means first wait 2sec, 2sec, 4sec, 4sec,....10sec
    wait= wait_exponential(multipier=1, min=2, max=10), 
    before_sleep= log_retry, #printing wait time as defined above
    reraise= True
)
def call_llm_backoff(prompt: str):
    return flaky_llm_call_backoff(prompt)

call_llm_backoff("Explain backoff retry")

In [ ]:
# Retry on specific exceptions
from tenacity import retry_if_exception_type

def log_retry_exception(retry_state: RetryCallState) -> None:
    wait_time = retry_state.next_action.sleep
    print(f"Attempt: {retry_state.attempt_number} failed -- retrying in {wait_time:.1f}s")

class InvalidRequestError(Exception):
    pass

@retry(
    stop= stop_after_attempt(5),
    wait= wait_fixed(1),
    retry= retry_if_exception_type((TimeoutError, InvalidRequestError)), # only retry these two
    before_sleep= log_retry_exception,
    reraise= True
)
def call_llm_custom_exception(prompt: str, error_type:type[Exception]= TimeoutError):
    if error_type:
        raise error_type("Simulated failure")
    return f"LLM response: {prompt}"

# A trainsent error retries 3 times then raises error
try:
    call_llm_custom_exception("test",error_type=TimeoutError)
except TimeoutError:
    print('Give up after retries on a TimeOutError')

try:
    call_llm_custom_exception("test",error_type=InvalidRequestError)
except InvalidRequestError:
    print('Failed immediately on InvalidRequestError -- no retries attempted')

In [ ]:
# Real LLM call with tenacity

@retry(
    stop= 5,
    wait_fixed= wait_exponential(multipler=1, min=2, max=10),
    retry= retry_if_exception_type((openai.RateLimitError, openai.APITimeOutError, openai.APIConnectionError)), # only retry these two
    before_sleep= log_retry_exception,
    reraise= True
)
def call_real_llm(prompt: str, client=openai.OpenAI):
    response = client.chat.completions.create(
        model= 'gpt-4o-mini',
        message= [{'role':'user','content':prompt}]
    )
    return response.choices[0].message.content

client= openai.OpenAI(api_key="sdfagsg")
print(call_real_llm('Say hello',client=client))

In [1]:
2+2

4

#### Testing web search tool (TavilySearch)

In [2]:
from components.tools.websearch_tool import internet_search

In [3]:
search = internet_search(query="what is the capital of France")

In [4]:
print(search)

{'query': 'what is the capital of France', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.britannica.com/place/France', 'title': 'France | History, Maps, Flag, Population, Cities, Capital, & Facts', 'content': "Historically, the Francien dialect became the official language in 1539, eventually replacing Latin and other dialects. Although regional languages were once discouraged, they have been reintroduced in some schools because several have maintained literary traditions. Today, French is considered one of the most internationally significant Romance languages.\n\nThe capital of France is Paris. Situated in the north-central part of the country, Paris is France's center of commerce and culture. [...] The capital and by far the most important city of France is Paris, one of the world’s preeminent cultural and commercial centres. A majestic city known as the ville lumière, or “city of light,” Paris has often been remade, most famously in the

In [5]:
print(search['query'])

what is the capital of France


In [10]:
print(search['results'][0]['content'])

Historically, the Francien dialect became the official language in 1539, eventually replacing Latin and other dialects. Although regional languages were once discouraged, they have been reintroduced in some schools because several have maintained literary traditions. Today, French is considered one of the most internationally significant Romance languages.

The capital of France is Paris. Situated in the north-central part of the country, Paris is France's center of commerce and culture. [...] The capital and by far the most important city of France is Paris, one of the world’s preeminent cultural and commercial centres. A majestic city known as the ville lumière, or “city of light,” Paris has often been remade, most famously in the mid-19th century under the command of Georges-Eugène, Baron Haussman, who was committed to Napoleon III’s vision of a modern city free of the choleric swamps and congested alleys of old, with broad avenues and a regular plan. Paris is now a sprawling [...] 

In [11]:
response = []
for i, r in enumerate(search["results"], 1):
    title   = r.get("title", "Unknown")
    url     = r.get("url", "")
    snippet = r.get("content", "").strip()
    # Keep only the first 300 characters to avoid wall-of-text
    if len(snippet) > 300:
        snippet = snippet[:300].rsplit(" ", 1)[0] + "..."

    response.append(f"{i}. **{title}**\n   {url}\n   {snippet}")

In [12]:
print(response)

['1. **France | History, Maps, Flag, Population, Cities, Capital, & Facts**\n   https://www.britannica.com/place/France\n   Historically, the Francien dialect became the official language in 1539, eventually replacing Latin and other dialects. Although regional languages were once discouraged, they have been reintroduced in some schools because several have maintained literary traditions. Today, French is considered one...', '2. **Paris facts: the capital of France in history**\n   https://home.adelphi.edu/~ca19535/page%204.html\n   |  |  |  |  |  |  |  |\n ---  ---  --- \n| Home | Spain | Sydney | San Francisco | Paris | Las Vegas | Maui |\n\n  \n Paris, France\n\n  \n\n## Paris facts: Paris, the capital of France\n\nParis is the capital of France, the largest country of Europe with 550 000 km2 (65 millions inhabitants).\n\nParis has...', '3. **Paris - Wikipedia**\n   https://en.wikipedia.org/wiki/Paris\n   Paris is the capital and largest city of France, with an estimated city popula